In [1]:
# --- Cell 1: Setup and load ---
import pandas as pd, numpy as np
from pathlib import Path
import plotly.express as px

BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd() / "data-analytics"
RAW, PROCESSED, DB = BASE / "data" / "raw", BASE / "data" / "processed", BASE / "sql" / "agriculture.db"
crop = pd.read_csv(PROCESSED / "cleaned_crop_data.csv")
merged = pd.read_csv(PROCESSED / "merged_yield_data.csv")
print(crop.shape, merged.shape)

(334630, 9) (298118, 15)


In [2]:
# --- Cell 2: Overview ---
print(f"{crop['state'].nunique()} states, {crop['district'].nunique()} districts, "
      f"{crop['crop'].nunique()} crops, years {crop['year'].min()}-{crop['year'].max()}")
crop.describe().round(2)

36 states, 707 districts, 55 crops, years 1997-2020


,year,area,production,yield
count,334630.00,334630.00,3.346300e+05,334630.00
mean,2008.87,11981.80,9.699751e+05,80.27
std,6.57,46490.24,2.170321e+07,913.79
min,1997.00,0.01,1.000000e+00,0.00
25%,2003.00,82.00,8.900000e+01,0.57
50%,2009.00,580.00,7.210000e+02,1.01
75%,2015.00,4348.00,7.200000e+03,2.42
max,2020.00,8580100.00,1.597800e+09,30093.33


In [3]:
# --- Cell 3: Top 10 producing states for a crop ---
CROP = "Rice"   # change to Wheat, Maize, Sugarcane ...
top = crop[crop.crop == CROP].groupby("state")["production"].sum().nlargest(10).reset_index()
px.bar(top, x="production", y="state", orientation="h", title=f"Top 10 {CROP}-producing states") \
  .update_layout(yaxis={"categoryorder": "total ascending"})

In [4]:
# --- Cell 4: Highest-yielding crops (Coconut excluded, different unit) ---
y = crop[crop.crop != "Coconut"].groupby("crop")["yield"].mean().nlargest(10).reset_index()
px.bar(y, x="crop", y="yield", title="Average yield by crop (tonnes/ha)")

In [5]:
# --- Cell 5: Yield trend over time ---
major = ["Rice", "Wheat", "Maize", "Potato"]
t = crop[crop.crop.isin(major)].groupby(["year", "crop"])["yield"].mean().reset_index()
px.line(t, x="year", y="yield", color="crop", markers=True, title="Average yield trend")

In [6]:
# --- Cell 6: Season-wise performance ---
s = crop.groupby("season").agg(records=("crop", "size"), area=("area", "sum"),
                               avg_yield=("yield", "median")).reset_index()
display(s)
px.bar(s, x="season", y="area", title="Cultivated area by season")

,season,records,area,avg_yield
0,Autumn,6854,5.867512e+07,1.106307
1,Kharif,134281,1.949352e+09,0.858804
2,Rabi,98437,1.329656e+09,0.981447
3,Summer,21787,9.778478e+07,1.598870
4,Whole Year,65085,2.932894e+08,3.994595
5,Winter,8186,2.807137e+08,1.400000


In [7]:
# --- Cell 7: Heatmap, yield by state and year ---
h = crop[crop.crop == "Wheat"].groupby(["state", "year"])["yield"].mean().unstack()
px.imshow(h, aspect="auto", color_continuous_scale="YlGn", title="Wheat yield: state x year")

In [8]:
# --- Cell 8: Rainfall vs yield ---
# Overall correlation is negative. This most likely does not mean rain hurts crops:
# low-rain irrigated states (Punjab, Haryana) have the best yields (confounding, not tested here).
for c in ["Rice", "Wheat", "Maize"]:
    x = merged[merged.crop == c]
    print(c, "correlation (rainfall, yield):", round(x["season_rainfall"].corr(x["yield"]), 3))

r = merged[(merged.crop == "Rice") & (merged.season == "Kharif")].groupby(["state", "year"]).agg(
    y=("yield", "mean"), rain=("season_rainfall", "mean")).reset_index()
px.scatter(r, x="rain", y="y", color="state", opacity=0.6, title="Kharif rice: monsoon rainfall vs yield")

Rice correlation (rainfall, yield): -0.287
Wheat correlation (rainfall, yield): -0.337
Maize correlation (rainfall, yield): -0.243


In [9]:
# --- Cell 9: Inside one state the relation is weak or mixed ---
for st in ["Punjab", "Maharashtra", "Uttar Pradesh", "Odisha", "Bihar"]:
    x = merged[(merged.state == st) & (merged.crop == "Rice") & (merged.season == "Kharif")]
    if len(x) > 30:
        print(st, round(x["season_rainfall"].corr(x["yield"]), 3))

Punjab 0.075
Maharashtra 0.069
Uttar Pradesh 0.013
Odisha 0.769
Bihar -0.684
